# SECTION 0: Imports

In [0]:
import requests
import json
from pyspark.sql import Row

# SECTION 1: Setup

In [0]:
%run ../config

# SECTION 2: Dataset-specific constants

In [0]:
BOUNDARIES_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.community_area_boundaries"

BOUNDARIES_DATASET_ID = "igwz-8jzy"
BOUNDARIES_ENDPOINT = f"https://{SOCRATA_DOMAIN}/api/v3/views/{BOUNDARIES_DATASET_ID}/query.json"
PAGE_SIZE_BOUNDARIES = 1000

BOUNDARY_COLUMNS = ["the_geom", "community", "area_numbe", "shape_area", "shape_len"]

# SECTION 3: Fetch Function

In [0]:
def fetch_boundaries():
    """
    Fetches all rows of the Boundaries - Community Areas dataset in a single request.
    """

    app_token = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

    headers = {
        "X-App-Token": app_token,
        "Content-Type": "application/json"
    }

    payload = {
        "query": "SELECT *",
        "page": {
            "pageNumber": 1,
            "pageSize": PAGE_SIZE_BOUNDARIES
        },
        "includeSynthetic": False
    }

    response = requests.post(
        BOUNDARIES_ENDPOINT, 
        headers=headers,
        json=payload,
        timeout=60
    )
    response.raise_for_status()
    rows = response.json()

    return rows

# SECTION 4: Normalize rows into a DataFrame

In [0]:
def normalize_row(row):
    """
    Converts every field to a string. the_geom is a nested JSON object, so it is serialized with json.dumps() rather than str() - str() on a dict produces Python's repr format (single-quoted), which cannot be parsed back as JSON later; json.dumps() keeps it as valid JSON text.
    """

    result = {}
    for col in BOUNDARY_COLUMNS:
        value = row.get(col)
        if value is None:
            result[col] = None
        elif col == "the_geom":
            result[col] = json.dumps(value)
        else:
            result[col] = str(value)
    return result

In [0]:
raw_rows = fetch_boundaries()
normalized_rows = [normalize_row(row) for row in raw_rows]

In [0]:
spark_rows = [Row(**row) for row in normalized_rows]
df = spark.createDataFrame(spark_rows)

# SECTION 5: Write to Bronze

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(BOUNDARIES_TABLE)

print(f"Loaded {df.count()} rows into {BOUNDARIES_TABLE}")